# 07 — Evaluation + ablations

1. Pass@1 on AIME-2024, AIME-2025, GPQA-Diamond (subset), LCB-v6 (subset).
2. TS-KL on a small oracle-tool-labeled probe.
3. Wall-clock training time to matched reward.
4. Ablations: η ∈ {0, 0.2, 1.0}; no-CF; predictor-fidelity stress test.


In [ ]:
import sys, os; sys.path.insert(0, str(os.path.abspath(os.path.join(os.getcwd(), '..'))))
import json, time, torch, random
from pathlib import Path
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from dyna_grpo.config import MODEL, PATHS, GRPO
from dyna_grpo.data import aime_2024, aime_2025, gpqa_diamond, livecodebench, extract_answer
from dyna_grpo.rewards import reward_for, reward_aime, reward_gpqa, reward_lcb
from dyna_grpo.metrics import pass_at_1, tool_selection_kl, per_tool_invocation_freq
from dyna_grpo.trace_collector import rollout
from dyna_grpo.tools import call_tool, parse_tool_call
from dyna_grpo.utils import save_metrics

In [ ]:
def load_actor(ckpt_dir: str | None = None):
    base = AutoModelForCausalLM.from_pretrained(
        MODEL.actor_name, torch_dtype=torch.bfloat16, device_map='cuda:0', trust_remote_code=True)
    if ckpt_dir:
        base = PeftModel.from_pretrained(base, ckpt_dir)
    base.eval()
    return base

def make_gen_fn(model, tok):
    def gen(ctx, max_new):
        enc = tok(ctx, return_tensors='pt', truncation=True, max_length=6000).to('cuda:0')
        with torch.no_grad():
            o = model.generate(**enc, max_new_tokens=max_new, do_sample=False,
                                pad_token_id=tok.pad_token_id)
        return tok.decode(o[0][enc.input_ids.size(1):], skip_special_tokens=True)
    return gen

In [ ]:
tok = AutoTokenizer.from_pretrained(MODEL.actor_name, trust_remote_code=True)
if tok.pad_token is None: tok.pad_token = tok.eos_token

def eval_aime(model, ds_fn, name, n=30, n_samples=4):
    ds = ds_fn()
    rewards_per = []
    for r in list(ds)[:n]:
        gen = make_gen_fn(model, tok)
        rs = []
        for _ in range(n_samples):
            t = rollout(r['problem'], gen, max_tool_calls=GRPO.max_tool_calls)
            rs.append(reward_aime(t.flat_text(), r['answer']))
        rewards_per.append(rs)
    p1 = pass_at_1(rewards_per)
    print(f'{name} pass@1 (n_samples={n_samples}): {p1:.3f}')
    return p1

def eval_lcb(model, n=30, n_samples=2):
    ds = livecodebench(limit=n)
    rewards_per = []
    for r in ds:
        gen = make_gen_fn(model, tok)
        rs = []
        prompt = r.get('question_content') or r.get('prompt') or r.get('text', '')
        for _ in range(n_samples):
            t = rollout(prompt, gen, max_tool_calls=GRPO.max_tool_calls)
            rs.append(reward_lcb(t.flat_text(), dict(r)))
        rewards_per.append(rs)
    p1 = pass_at_1(rewards_per)
    print(f'LCB-v6 pass@1: {p1:.3f}')
    return p1

def eval_gpqa(model, n=30, n_samples=4):
    ds = gpqa_diamond(limit=n)
    rewards_per = []
    for r in ds:
        gen = make_gen_fn(model, tok)
        rs = []
        for _ in range(n_samples):
            t = rollout(r.get('Question', ''), gen, max_tool_calls=GRPO.max_tool_calls)
            rs.append(reward_gpqa(t.flat_text(), r.get('Correct Answer', ''),
                                    [r.get(f'Incorrect Answer {i}', '') for i in (1,2,3)]))
        rewards_per.append(rs)
    p1 = pass_at_1(rewards_per)
    print(f'GPQA-Diamond pass@1: {p1:.3f}')
    return p1

In [ ]:
# Sweep all checkpoints (zero-shot, baseline, dyna-no-cf, dyna-full)
results = {}
RUNS = {
    'zero_shot': None,
    'baseline_grpo': str(Path(PATHS['ckpts']) / 'baseline_grpo' / 'final'),
    'dyna_grpo_full': str(Path(PATHS['ckpts']) / 'dyna_grpo_full' / 'final'),
}
for name, ckpt in RUNS.items():
    print(f'\n=== Eval: {name} ===')
    if ckpt and not Path(ckpt).exists():
        print(f'  Skipping; ckpt not found: {ckpt}')
        continue
    m = load_actor(ckpt)
    results[name] = {
        'aime_2024': eval_aime(m, aime_2024, 'AIME 2024', n=30, n_samples=4),
        'aime_2025': eval_aime(m, aime_2025, 'AIME 2025', n=15, n_samples=4),
        'gpqa':       eval_gpqa(m, n=30, n_samples=2),
        'lcb':        eval_lcb(m, n=30, n_samples=2),
    }
    del m; torch.cuda.empty_cache()
save_metrics(Path(PATHS['logs']) / 'eval_results.json', results)
results

In [ ]:
# Wall-clock-to-target-reward (read from training history jsonl)
import json
def wallclock_to_target(history_path: str, target: float) -> float | None:
    obj = json.loads(Path(history_path).read_text())
    for h in obj['history']:
        if h.get('reward_mean', 0) >= target:
            return h.get('wallclock_s')
    return None

for run in ('baseline_grpo', 'dyna_grpo_full'):
    p = Path(PATHS['logs']) / f'{run}.jsonl'
    if p.exists():
        for tgt in (0.3, 0.4, 0.5):
            t = wallclock_to_target(str(p), tgt)
            print(f'{run} -> reward {tgt}: {t}s')

In [ ]:
# TS-KL: oracle-tool-labeled small probe (we hand-label or use heuristics)
ORACLE_PROBE = [
    {'prompt': 'What is 137 * 421?', 'oracle': 'calc'},
    {'prompt': 'Who won the FIFA World Cup in 2022?', 'oracle': 'search'},
    {'prompt': 'Write Python to find primes up to 100.', 'oracle': 'code'},
    {'prompt': 'What is the integral of sin(x) from 0 to pi?', 'oracle': 'calc'},
    {'prompt': 'List the latest CVPR best paper.', 'oracle': 'search'},
    {'prompt': 'Solve x^2 + 5x - 6 = 0.', 'oracle': 'calc'},
    {'prompt': 'Reverse a linked list in Python.', 'oracle': 'code'},
]

def predict_first_tool(model, prompt: str) -> str:
    gen = make_gen_fn(model, tok)
    t = rollout(prompt, gen, max_tool_calls=1, max_new_tokens=512)
    for s in t.segments:
        if s.type == 'tool_call':
            return s.tool
    return 'none'

ts_kl = {}
for name, ckpt in RUNS.items():
    if ckpt and not Path(ckpt).exists(): continue
    m = load_actor(ckpt)
    preds = [predict_first_tool(m, p['prompt']) for p in ORACLE_PROBE]
    golds = [p['oracle'] for p in ORACLE_PROBE]
    ts_kl[name] = tool_selection_kl(preds, golds)
    del m; torch.cuda.empty_cache()
print('TS-KL:', ts_kl)
save_metrics(Path(PATHS['logs']) / 'ts_kl.json', ts_kl)

## Ablations
Re-run notebook 06 with these envs/flags:
- **A1 (no CF)**: set `USE_CF=False`, `RUN_NAME='dyna_no_cf'`
- **A2 (η=0)**: set `ETA=0`, `RUN_NAME='dyna_eta0'`
- **A3 (η=1)**: set `ETA=1.0`, `RUN_NAME='dyna_eta1'`  (recovers vanilla GRPO with predictors loaded but unused)

Then re-run the eval cells above with those checkpoints added to `RUNS`.
